In [18]:
%env mlflow_tracking_username = dan
%env mlflow_tracking_password = DanLarson

env: mlflow_tracking_username=dan
env: mlflow_tracking_password=DanLarson


In [1]:
## 1.1 Import Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, recall_score, precision_score, f1_score
import matplotlib.pyplot as plt
from sklearn import tree
import mlflow
import mlflow.sklearn
import time
import seaborn as sns
import sweetviz as sv


In [23]:
## Config MLFlow

%env mlflow_tracking_username = 'dan'
%env mlflow_tracking_password = 'DanLarson'
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000/")
mlflow.set_experiment("time_based_data_prep")
mlflow.enable_system_metrics_logging() 

env: mlflow_tracking_username='dan'
env: mlflow_tracking_password='DanLarson'


## Data Preperation

In [3]:
## Import the data
import duckdb as db
time_df = db.sql("select * from read_parquet('C:/MLAR_DATA/T0001/FS_turnaround_time_table/TIME_BASED_*.parquet')").to_df()
labelled_data = db.sql("select SOURCE_DB, lower(audit_table_name) as audit_table_name, audit_table_key, max(case when severity in ('1','2','3','4','5') then 1 else 0 end) as anomaly from read_parquet('C:/MLAR_DATA/T0001/labelled_*.parquet') group by 1,2,3").to_df()
time_with_label = time_df.merge(labelled_data,left_on=['table_key','table_name','SOURCE_DB'] ,right_on=['audit_table_key','audit_table_name','SOURCE_DB'],how='left')
time_with_label['anomaly'] = time_with_label['anomaly'].fillna(0)

print(time_with_label.head())
print(time_with_label.describe())
print(time_with_label['anomaly'].value_counts())

   table_key table_name status          changed_on          login_date  \
0      19307     sample      A 2023-01-31 12:35:26 2022-01-27 15:07:38   
1      19316     sample      A 2023-01-31 12:35:26 2022-01-27 15:07:45   
2      37791     sample      U 2022-10-24 00:00:08 2022-10-24 00:00:07   
3      39268     sample      U 2022-12-07 00:00:09 2022-12-07 00:00:08   
4      38580     sample      A 2023-02-02 10:29:02 2022-11-14 15:53:01   

         sampled_date           recd_date        date_started  \
0 2022-01-31 15:55:58 2022-02-09 15:30:08 2022-02-09 15:31:23   
1 2022-01-31 15:55:58 2022-02-01 14:53:39 2022-02-07 13:49:55   
2                 NaT                 NaT                 NaT   
3                 NaT                 NaT                 NaT   
4 2022-11-17 15:45:41 2022-11-21 13:50:17 2022-11-21 14:47:28   

       date_completed       date_reviewed  ...  date_reviewed_audit_log  \
0 2022-02-09 15:31:23 2023-01-31 12:35:02  ...      2023-01-31 12:35:02   
1 2022-02-07 1

In [4]:
## Adding some new features

# Extract day of the week
time_with_label['started_day_of_week'] = time_with_label['date_started'].dt.dayofweek  # Monday=0, Sunday=6
# Alternative: df['day_name'] = df['date_column'].dt.day_name()

# Extract hour
time_with_label['started_hour']= time_with_label['date_started'].dt.hour

# Extract week of the year
time_with_label['started_week_of_year'] = time_with_label['date_started'].dt.isocalendar().week


In [5]:
## 2.3 Data Preprocessing
features = ['status', 'login_to_recieve_ms', 'recd_to_sample_ms', 'recd_to_started_ms',
            'recd_to_completed_ms', 'started_to_completed_ms', 'recd_to_revewed_ms',
            'started_to_revewed_ms', 'completed_to_revewed_ms', 'login_to_changed_ms',
            'recd_to_changed_ms', 'started_to_changed_ms', 'completed_to_changed_ms',
            'reviewed_to_changed_ms', 'login_date_compare', 'recd_date_compare',
            'DATE_STARTED_compare', 'date_completed_compare', 'date_reviewed_compare',
            'started_day_of_week', 'started_hour','started_week_of_year']
target = 'anomaly'

In [6]:
## Check for missing values

print(time_with_label[features].isnull().sum())

status                          0
login_to_recieve_ms        133612
recd_to_sample_ms          322374
recd_to_started_ms         168437
recd_to_completed_ms       222199
started_to_completed_ms    222448
recd_to_revewed_ms         255309
started_to_revewed_ms      255311
completed_to_revewed_ms    255260
login_to_changed_ms             0
recd_to_changed_ms         133612
started_to_changed_ms      168389
completed_to_changed_ms    222150
reviewed_to_changed_ms     255260
login_date_compare              0
recd_date_compare               0
DATE_STARTED_compare            0
date_completed_compare          0
date_reviewed_compare           0
started_day_of_week        168389
started_hour               168389
started_week_of_year       168389
dtype: int64


In [7]:
from sklearn.impute import SimpleImputer
import numpy as np
import pandas as pd

# Define the columns
numerical_cols = ['login_to_recieve_ms', 'recd_to_sample_ms', 'recd_to_started_ms',
            'recd_to_completed_ms', 'started_to_completed_ms', 'recd_to_revewed_ms',
            'started_to_revewed_ms', 'completed_to_revewed_ms', 'login_to_changed_ms',
            'recd_to_changed_ms', 'started_to_changed_ms', 'completed_to_changed_ms',
            'reviewed_to_changed_ms',
            'started_day_of_week', 'started_hour','started_week_of_year']
categorical_cols = ['status']

# Assuming 'customer_id' is the column identifying customers
customer_id_col = 'SOURCE_DB'

def impute_customer_level(df, cols, customer_id_col, strategy='mean'):
    imputed_df = df.copy()
    for customer_id in df[customer_id_col].unique():
        customer_data = df[df[customer_id_col] == customer_id]
        imputer = SimpleImputer(strategy=strategy)
        for col in cols:
            # Convert column to float64 before imputation
            customer_data[col] = customer_data[col].astype(float)
            imputed_df.loc[imputed_df[customer_id_col] == customer_id, col] = imputer.fit_transform(customer_data[[col]])
    return imputed_df

# Convert numerical columns to float64
time_with_label[numerical_cols] = time_with_label[numerical_cols].astype(float)

# Impute numerical columns
time_with_label = impute_customer_level(time_with_label, numerical_cols, customer_id_col, strategy='mean')

# Impute categorical columns
#time_with_label = impute_customer_level(time_with_label, categorical_cols, customer_id_col, strategy='most_frequent')

C:\Users\dan.larson\AppData\Local\Temp\6\ipykernel_14092\1890668555.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  customer_data[col] = customer_data[col].astype(float)
C:\Users\dan.larson\AppData\Local\Temp\6\ipykernel_14092\1890668555.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  customer_data[col] = customer_data[col].astype(float)
C:\Users\dan.larson\AppData\Local\Temp\6\ipykernel_14092\1890668555.py:24: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFr

In [8]:
### Transform the time based data since it is all postively skewed

import numpy as np

# List of features to be log-transformed
features_to_log_transform = [
    'login_to_recieve_ms', 'recd_to_sample_ms', 'recd_to_started_ms',
    'recd_to_completed_ms', 'started_to_completed_ms', 'recd_to_revewed_ms',
    'started_to_revewed_ms', 'completed_to_revewed_ms', 'login_to_changed_ms',
    'recd_to_changed_ms', 'started_to_changed_ms', 'completed_to_changed_ms',
    'reviewed_to_changed_ms'
]

# Apply log1p transformation to each feature
for feature in features_to_log_transform:
    # Ensure there are no negative values, replace with 1 if negative
    time_with_label[feature] = time_with_label[feature].apply(lambda x: max(x, 1))
    # Apply log1p transformation
    time_with_label[feature] = np.log1p(time_with_label[feature])

# Check the transformation
print(time_with_label[[f for f in features_to_log_transform]].head())

   login_to_recieve_ms  recd_to_sample_ms  recd_to_started_ms  \
0            20.840649           0.693147           11.225257   
1            19.881976           0.693147           20.058854   
2            21.138213           0.693147           20.388376   
3            21.138213           0.693147           20.388376   
4            20.208158           0.693147           15.048363   

   recd_to_completed_ms  started_to_completed_ms  recd_to_revewed_ms  \
0             11.225257                 0.693147           24.149087   
1             20.058854                 0.693147           24.171388   
2             21.858877                21.578433           22.487745   
3             21.858877                21.578433           22.487745   
4             15.048363                 0.693147           22.563038   

   started_to_revewed_ms  completed_to_revewed_ms  login_to_changed_ms  \
0              24.149085                24.149085            24.185008   
1              24.154886    

In [9]:
from sklearn.preprocessing import RobustScaler

numerical_cols = [ 'login_to_recieve_ms', 'recd_to_sample_ms', 'recd_to_started_ms',
    'recd_to_completed_ms', 'started_to_completed_ms', 'recd_to_revewed_ms',
    'started_to_revewed_ms', 'completed_to_revewed_ms', 'login_to_changed_ms',
    'recd_to_changed_ms', 'started_to_changed_ms', 'completed_to_changed_ms',
    'reviewed_to_changed_ms']

# Initialize the MinMaxScaler
scaler = RobustScaler()

# Apply MinMax normalization for each customer
df_normalized = time_with_label.groupby('SOURCE_DB').apply(
    lambda group: pd.DataFrame(scaler.fit_transform(group[numerical_cols]), columns=numerical_cols, index=group.index)
).reset_index(drop=True)

# Combine normalized data back with non-normalized columns
time_with_label.update(df_normalized)


In [10]:

time_with_label.groupby('SOURCE_DB')['recd_to_sample_ms'].describe()

,count,mean,std,min,25%,50%,75%,max
SOURCE_DB,,,,,,,,
X5,155437.0,0.007544,0.341826,0.0,0.0,0.0,0.0,17.108157
X7,194219.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.000000
X8,183435.0,0.020273,0.633585,0.0,0.0,0.0,0.0,23.078511


In [11]:
## Optional EDA report on the data 


my_report = sv.analyze(time_with_label)
my_report.show_html(filepath='TIME_based_feature_analysis.html') # Default arguments will generate to "SWEETVIZ_REPORT.html"


                                             |          | [  0%]   00:00 -> (? left)

Report TIME_based_feature_analysis.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [27]:
my_report = sv.analyze(time_with_label,target_feat='anomaly')
my_report.show_html(filepath='TIME_with_label_V1.html') # Default arguments will generate to "SWEETVIZ_REPORT.html"


                                             |          | [  0%]   00:00 -> (? left)

Report TIME_with_label_V1.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [12]:
## View of data

time_with_label.head()

,table_key,table_name,status,changed_on,login_date,sampled_date,recd_date,date_started,date_completed,date_reviewed,...,DATE_STARTED_compare,date_completed_compare,date_reviewed_compare,SOURCE_DB,audit_table_name,audit_table_key,anomaly,started_day_of_week,started_hour,started_week_of_year
0,19307,sample,A,2023-01-31 12:35:26,2022-01-27 15:07:38,2022-01-31 15:55:58,2022-02-09 15:30:08,2022-02-09 15:31:23,2022-02-09 15:31:23,2023-01-31 12:35:02,...,True,True,True,X5,NaN,NaN,0.0,2.000000,15.000000,6.000000
1,19316,sample,A,2023-01-31 12:35:26,2022-01-27 15:07:45,2022-01-31 15:55:58,2022-02-01 14:53:39,2022-02-07 13:49:55,2022-02-07 13:49:55,2023-01-31 12:35:02,...,True,True,True,X5,NaN,NaN,0.0,0.000000,13.000000,6.000000
2,37791,sample,U,2022-10-24 00:00:08,2022-10-24 00:00:07,NaT,NaT,NaT,NaT,NaT,...,False,False,False,X5,NaN,NaN,0.0,2.077798,12.412472,21.303209
3,39268,sample,U,2022-12-07 00:00:09,2022-12-07 00:00:08,NaT,NaT,NaT,NaT,NaT,...,False,False,False,X5,NaN,NaN,0.0,2.077798,12.412472,21.303209
4,38580,sample,A,2023-02-02 10:29:02,2022-11-14 15:53:01,2022-11-17 15:45:41,2022-11-21 13:50:17,2022-11-21 14:47:28,2022-11-21 14:47:28,2023-02-02 10:28:39,...,True,True,True,X5,NaN,NaN,0.0,0.000000,14.000000,47.000000


In [ ]:
# One-Hot Encode categorical variables
X = pd.get_dummies(time_with_label[features], columns=['status'])
y = time_with_label[target]

## Base Model 
Building model on all three customers to determine a candadite model. 

### Success Criteria
A model with 
- .90 recall (captures 90% of actual rare events), 
- accuracy >98%, 
- precision > 75%

### Things to try
- Oversamplig with SMOTE
- Optimize the threshold
- Review false negatives to see if they should be considered rare events

#### Notes: 
- Model falls apart when doing holdout.
- Attempted to normalize the data using a normalization model
    - minimal impact





In [ ]:
mlflow.enable_system_metrics_logging() 

In [ ]:

## 3.2 Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 3.3 Initialize and Train Model
clf = DecisionTreeClassifier(random_state=42)

## 3.4 Evaluate Model
start_time = time.time()
clf.fit(X_train, y_train)
end_time = time.time()
training_time = end_time - start_time

y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
class_report = classification_report(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

In [ ]:

print(f"Training Time: {training_time} seconds")
print("Accuracy:", accuracy)
print("Classification Report:")
print(class_report)
print("Confusion Matrix:")
print(conf_matrix)


In [ ]:

## 4.1 Start MLflow Run
with mlflow.start_run():
    
    ## 4.2 Log Parameters and Metrics
    mlflow.log_param("model_type", "DecisionTreeClassifier")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("sample Method","train_test")
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("f1 score", f1)
    mlflow.log_metric("training_time", training_time)
    
    ## 4.3 Log Model and Artifacts
    mlflow.sklearn.log_model(clf, "model")
    
    # Visualize and log the decision tree
    plt.figure(figsize=(20,10))
    tree.plot_tree(clf, feature_names=X.columns, class_names=['Not Anomaly', 'Anomaly'], filled=True)
    plt.savefig("decision_tree.png")
    mlflow.log_artifact("decision_tree.png")

        # Plot feature importances
    feature_importances = clf.feature_importances_
    feature_names = X.columns
    importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances})
    importance_df = importance_df.sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(10, 8))
    sns.barplot(x='Importance', y='Feature', data=importance_df)
    plt.title('Feature Importance')
    plt.tight_layout()
    plt.savefig("feature_importance.png")
    mlflow.log_artifact("feature_importance.png")

## SMOTE (Synthetic Minority Over-sampling Technique)
Definition: SMOTE is a method used to address class imbalance in datasets by generating synthetic samples for the minority class.

### How it Works:

It creates new, synthetic samples by interpolating between existing minority class samples.

For each minority class sample, it selects one or more of its nearest neighbors.

Synthetic samples are generated along the line segments joining the original minority sample and its selected neighbors.

### Why Use SMOTE:

Balances Class Distribution: Helps to mitigate the bias towards the majority class in imbalanced datasets.

Improves Model Performance: By providing the model with more examples of the minority class, it can learn better and improve metrics like recall and F1-score.

Avoids Overfitting: Unlike simple oversampling (which duplicates minority class samples), SMOTE generates new, unique samples, reducing the risk of overfitting.


In [ ]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"  # Set this to the number of cores you want to use

from imblearn.over_sampling import SMOTE

In [ ]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

In [ ]:
clf = DecisionTreeClassifier(random_state=42)

## 3.5 Evaluate Model
start_time = time.time()
clf.fit(X_train_smote, y_train_smote)
end_time = time.time()
training_time = end_time - start_time

y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
class_report = classification_report(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

In [ ]:
print(f"Training Time: {training_time} seconds")
print("Accuracy:", accuracy)
print("Classification Report:")
print(class_report)
print("Confusion Matrix:")
print(conf_matrix)


In [ ]:
## 4.1 Start MLflow Run
with mlflow.start_run():
    
    ## 4.2 Log Parameters and Metrics
    mlflow.log_param("model_type", "DecisionTreeClassifier")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("sample Method","SMOTE")
    mlflow.log_metric("accuracy", accuracy)

    mlflow.log_metric("recall", recall)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("f1 score", f1)
    mlflow.log_metric("training_time", training_time)
    
    ## 4.4 Log Model and Artifacts
    mlflow.sklearn.log_model(clf, "model")
    
    # Visualize and log the decision tree
    plt.figure(figsize=(20,10))
    tree.plot_tree(clf, feature_names=X.columns, class_names=['Not Anomaly', 'Anomaly'], filled=True)
    plt.savefig("decision_tree.png")
    mlflow.log_artifact("decision_tree.png")

    # Plot feature importances
    feature_importances = clf.feature_importances_
    feature_names = X.columns
    importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances})
    importance_df = importance_df.sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(10, 8))
    sns.barplot(x='Importance', y='Feature', data=importance_df)
    plt.title('Feature Importance')
    plt.tight_layout()
    plt.savefig("feature_importance.png")
    mlflow.log_artifact("feature_importance.png")

In [ ]:
## SMOTE with Random Forest Method
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_recall_curve


clf = RandomForestClassifier(n_estimators=500,  # Number of trees
    max_depth=10,  # Maximum depth of each tree
    min_samples_split=5,  # Minimum samples required to split an internal node
    min_samples_leaf=4,  # Minimum samples required to be at a leaf node
    max_features='sqrt',  # Number of features to consider when looking for the best split
    bootstrap=True,  # Whether bootstrap samples are used when building trees
    class_weight='balanced',  # Weights associated with classes
    random_state=42  # Random seed for reproducibility
    )

## 3.5 Evaluate Model
start_time = time.time()
clf.fit(X_train_smote, y_train_smote)
end_time = time.time()
training_time = end_time - start_time

y_pred = clf.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
class_report = classification_report(y_test, y_pred)
conf_matrix = confusion_matrix(y_test, y_pred)

In [ ]:
print(f"Training Time: {training_time} seconds")
print("Accuracy:", accuracy)
print("Classification Report:")
print(class_report)
print("Confusion Matrix:")
print(conf_matrix)

In [ ]:
recall

In [ ]:
## 4.1 Start MLflow Run
with mlflow.start_run():
    
    ## 4.2 Log Parameters and Metrics
    mlflow.log_param("model_type", "randomForest")
    mlflow.log_param("random_state", 42)
    mlflow.log_param("sample Method","SMOTE")
    mlflow.log_metric("accuracy", accuracy)

    mlflow.log_metric("recall", recall)
    mlflow.log_metric("precision", precision)
    mlflow.log_metric("f1 score", f1)
    mlflow.log_metric("training_time", training_time)
    
    ## 4.4 Log Model and Artifacts
    mlflow.sklearn.log_model(clf, "model")
    
    # Plot feature importances
    feature_importances = clf.feature_importances_

    feature_names = X.columns
    importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': feature_importances})
    importance_df = importance_df.sort_values(by='Importance', ascending=False)
    
    plt.figure(figsize=(10, 8))
    sns.barplot(x='Importance', y='Feature', data=importance_df)
    plt.title('Feature Importance')
    plt.tight_layout()
    plt.savefig("feature_importance.png")
    mlflow.log_artifact("feature_importance.png")

In [13]:

customerList = ["X5", "X7","X8"]#,"X3","X4","X1","X2","X6"] #,"X7","X8

In [14]:
time_with_label['SOURCE_DB']

0         X5
1         X5
2         X5
3         X5
4         X5
          ..
533086    X8
533087    X8
533088    X8
533089    X8
533090    X8
Name: SOURCE_DB, Length: 533091, dtype: object

In [24]:
## Set auto logging
mlflow.enable_system_metrics_logging()
features = ['status', 'login_to_recieve_ms', 'recd_to_sample_ms', 'recd_to_started_ms',
            'recd_to_completed_ms', 'started_to_completed_ms', 'recd_to_revewed_ms',
            'started_to_revewed_ms', 'completed_to_revewed_ms', 'login_to_changed_ms',
            'recd_to_changed_ms', 'started_to_changed_ms', 'completed_to_changed_ms',
            'reviewed_to_changed_ms', 'login_date_compare', 'recd_date_compare',
            'DATE_STARTED_compare', 'date_completed_compare', 'date_reviewed_compare',
            'started_day_of_week', 'started_hour','started_week_of_year']

with mlflow.start_run(run_name="parent_run_sklearn_decision_tree_1"):
    for customer in customerList:
    # Split data into training and holdout sets
        holdout_data = time_with_label[time_with_label['SOURCE_DB'] == customer]
        training_data = time_with_label[time_with_label['SOURCE_DB'] != customer]

        # One-Hot Encode categorical variables
        X_train = pd.get_dummies(training_data[features], columns=['status'])
        y_train = training_data[target]
        X_holdout = pd.get_dummies(holdout_data[features], columns=['status'])
        y_holdout = holdout_data[target]

        # Train model on training data
        with mlflow.start_run(run_name=f"child_holdout_{customer}_run_sklearn_decision_tree_1", nested=True):
            clf = DecisionTreeClassifier(random_state=42)
            clf.fit(X_train, y_train)

            # Evaluate model on holdout data
            y_pred = clf.predict(X_holdout)
            accuracy = accuracy_score(y_holdout, y_pred)
            recall = recall_score(y_holdout, y_pred)
            precision = precision_score(y_holdout, y_pred)
            f1 = f1_score(y_holdout, y_pred)

            # Log metrics and artifacts
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("f1 score", f1)
            mlflow.log_artifact("feature_importance.png")
            

2024/06/18 13:24:45 WARNING mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics because creating `GPUMonitor` failed with error: `pynvml` is not installed, to log GPU metrics please run `pip install pynvml` to install it..
2024/06/18 13:24:45 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2024/06/18 13:24:46 WARNING mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics because creating `GPUMonitor` failed with error: `pynvml` is not installed, to log GPU metrics please run `pip install pynvml` to install it..
2024/06/18 13:24:46 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2024/06/18 13:25:11 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2024/06/18 13:25:11 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
2024/06/18 13:25:12 WARNING mlflow.system_metrics.system_metrics_monitor:

In [26]:
## Set auto logging

from imblearn.over_sampling import SMOTE

mlflow.enable_system_metrics_logging()
features = ['status', 'login_to_recieve_ms', 'recd_to_sample_ms', 'recd_to_started_ms',
            'recd_to_completed_ms', 'started_to_completed_ms', 'recd_to_revewed_ms',
            'started_to_revewed_ms', 'completed_to_revewed_ms', 'login_to_changed_ms',
            'recd_to_changed_ms', 'started_to_changed_ms', 'completed_to_changed_ms',
            'reviewed_to_changed_ms', 'login_date_compare', 'recd_date_compare',
            'DATE_STARTED_compare', 'date_completed_compare', 'date_reviewed_compare',
            'started_day_of_week', 'started_hour','started_week_of_year']

with mlflow.start_run(run_name="parent_run_sklearn_decision_tree_1"):
    for customer in customerList:
    # Split data into training and holdout sets
        holdout_data = time_with_label[time_with_label['SOURCE_DB'] == customer]
        training_data = time_with_label[time_with_label['SOURCE_DB'] != customer]

        # One-Hot Encode categorical variables
        X_train = pd.get_dummies(training_data[features], columns=['status'])
        y_train = training_data[target]
        X_holdout = pd.get_dummies(holdout_data[features], columns=['status'])
        y_holdout = holdout_data[target]
        smote = SMOTE(random_state=42)
        X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

        # Train model on training data
        with mlflow.start_run(run_name=f"child_holdout_{customer}_run_sklearn_decision_tree_1", nested=True):
            clf = DecisionTreeClassifier(random_state=42)
            clf.fit(X_train_smote, y_train_smote)

            # Evaluate model on holdout data
            y_pred = clf.predict(X_holdout)
            accuracy = accuracy_score(y_holdout, y_pred)
            recall = recall_score(y_holdout, y_pred)
            precision = precision_score(y_holdout, y_pred)
            f1 = f1_score(y_holdout, y_pred)

            # Log metrics and artifacts
            mlflow.log_metric("accuracy", accuracy)
            mlflow.log_metric("recall", recall)
            mlflow.log_metric("precision", precision)
            mlflow.log_metric("f1 score", f1)
            mlflow.log_artifact("feature_importance.png")

2024/06/18 14:44:45 WARNING mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics because creating `GPUMonitor` failed with error: `pynvml` is not installed, to log GPU metrics please run `pip install pynvml` to install it..
2024/06/18 14:44:45 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2024/06/18 14:44:46 WARNING mlflow.sklearn: Training metrics will not be recorded because training labels were not specified. To automatically record training metrics, provide training labels as inputs to the model training function.
2024/06/18 14:44:46 WARNING mlflow.sklearn: Failed to infer model signature: the trained model does not have a `predict` or `transform` function, which is required in order to infer the signature
2024/06/18 14:44:46 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!
c:\Users\dan.larson\AppData\Local\Programs\Python\Python311\Lib\site-packages\joblib\externals\loky\backe

## Analysis to inform the modeling. 



In [ ]:

time_with_label.groupby('SOURCE_DB')['recd_to_sample_ms'].describe()


In [ ]:
time_with_label

In [ ]:
df_normalized

In [ ]:
# Create a boxplot to see the distribution of login_to_receive_ms by SOURCE_DB
plt.figure(figsize=(10, 6))  # Set the figure size for better visibility
sns.boxplot(x='SOURCE_DB', y='login_to_recieve_ms', data=time_with_label)
plt.title('Distribution of Login to Receive MS by SOURCE_DB')
plt.xlabel('Source Database (Customer)')
plt.ylabel('Login to Receive MS (Milliseconds)')
plt.show()

# Alternatively, use a violinplot for a more detailed distribution comparison
plt.figure(figsize=(10, 6))
sns.violinplot(x='SOURCE_DB', y='login_to_recieve_ms', data=time_with_label)
plt.title('Detailed Distribution of Login to Receive MS by SOURCE_DB')
plt.xlabel('Source Database (Customer)')
plt.ylabel('Login to Receive MS (Milliseconds)')
plt.show()

In [ ]:
time_with_label['login_to_recieve_ms']

## Question
Is the current labeled data practical for these purposes. What is the overlap with a EIF model? 